# Ensemble — XLM-R + Malay BERT
Combines predictions from both fine-tuned transformers by averaging probability distributions, evaluated against the gold test set.

In [2]:
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report, confusion_matrix

XLMR_PATH = "../models/transformers/bestfinedtuned_trans_xmlr"
MALAY_BERT_PATH = "../models/transformers/malay_bert_finetuned"

xlmr_tok = AutoTokenizer.from_pretrained(XLMR_PATH)
xlmr_model = AutoModelForSequenceClassification.from_pretrained(XLMR_PATH)
xlmr_model.eval()

malay_tok = AutoTokenizer.from_pretrained(MALAY_BERT_PATH)
malay_model = AutoModelForSequenceClassification.from_pretrained(MALAY_BERT_PATH)
malay_model.eval()

id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
print('Both models loaded')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both models loaded


## Load Gold Test Set

In [3]:
test_df = pd.read_csv('../data/NLP_Dataset_preparation/gold_labeled.csv')
y_test = test_df['human_label'].tolist()
texts = test_df['body'].astype(str).tolist()
labels = ['negative', 'neutral', 'positive']
print(f'Test rows: {len(test_df)}')

Test rows: 402


## Ensemble Prediction Function

In [4]:
def get_probs(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
    return torch.softmax(logits, dim=1)[0]

def ensemble_predict(text, xlmr_weight=0.5):
    p1 = get_probs(text, xlmr_tok, xlmr_model)
    p2 = get_probs(text, malay_tok, malay_model)
    combined = xlmr_weight * p1 + (1 - xlmr_weight) * p2
    pred_id = torch.argmax(combined).item()
    return id2label[pred_id]

## Test Three Weight Splits (favor XLM-R, equal, favor Malay BERT)

In [5]:
import os
results = []

for xlmr_weight in [0.7, 0.5, 0.3]:
    preds = [ensemble_predict(t, xlmr_weight) for t in texts]
    acc = accuracy_score(y_test, preds)
    kappa = cohen_kappa_score(y_test, preds)
    report = classification_report(y_test, preds, target_names=labels, output_dict=True)
    print(f'\nXLM-R weight={xlmr_weight}, Malay BERT weight={1-xlmr_weight}')
    print(f'Accuracy: {acc*100:.1f}%   Kappa: {kappa:.3f}')
    print(classification_report(y_test, preds, target_names=labels))
    results.append({
        'model': f'Ensemble (XLMR {xlmr_weight} / MalayBERT {1-xlmr_weight})',
        'status': 'ensemble',
        'accuracy': acc,
        'kappa': kappa,
        'neg_f1': report['negative']['f1-score'],
        'neu_f1': report['neutral']['f1-score'],
        'pos_f1': report['positive']['f1-score'],
    })

best = max(results, key=lambda r: r['kappa'])
print(f"\nBest ensemble: {best['model']} (kappa={best['kappa']:.3f})")

RESULTS_FILE = '../results/model_results.csv'
new_rows = pd.DataFrame(results)
existing = pd.read_csv(RESULTS_FILE)
existing = existing[~existing['model'].isin(new_rows['model'])]
combined = pd.concat([existing, new_rows], ignore_index=True)
combined.to_csv(RESULTS_FILE, index=False)
print(combined.to_string(index=False))


XLM-R weight=0.7, Malay BERT weight=0.30000000000000004
Accuracy: 67.9%   Kappa: 0.361
              precision    recall  f1-score   support

    negative       0.60      0.49      0.54       124
     neutral       0.72      0.83      0.77       241
    positive       0.50      0.30      0.37        37

    accuracy                           0.68       402
   macro avg       0.61      0.54      0.56       402
weighted avg       0.66      0.68      0.67       402


XLM-R weight=0.5, Malay BERT weight=0.5
Accuracy: 69.2%   Kappa: 0.375
              precision    recall  f1-score   support

    negative       0.64      0.48      0.55       124
     neutral       0.72      0.86      0.78       241
    positive       0.55      0.30      0.39        37

    accuracy                           0.69       402
   macro avg       0.64      0.55      0.57       402
weighted avg       0.68      0.69      0.67       402


XLM-R weight=0.3, Malay BERT weight=0.7
Accuracy: 70.9%   Kappa: 0.405
      